# Comprehensive Bot Question Tracking
**Date:** 2026-03-21

Combines HTML score table parsing with Metaculus API enrichment into a single workflow.

- **HTML source**: Question number, title, coverage, score, question weight
- **API source**: My forecast, open date, resolution date, resolution value, question type
- **Output**: Single merged CSV

In [1]:
# === CONFIG (all inputs here) ===
from pathlib import Path

HTML_FILE = Path(r"C:\Users\Donni\projects\metac_bot_Spring_2026\data\Spring 2026 FutureEval Bot Tournament 03-21-2026.html")
OUTPUT_CSV = Path(r"C:\Users\Donni\projects\metac_bot_Spring_2026\products\Bot_Question_Tracking_2026-03-21_v01.csv")

import os
METACULUS_TOKEN = os.getenv('METACULUS_BOT_API_TOKEN')
RATE_LIMIT_DELAY = 2.0

print(f"HTML: {HTML_FILE.name}")
print(f"Output: {OUTPUT_CSV.name}")
print(f"Auth: {'Token present' if METACULUS_TOKEN else 'NO TOKEN - set METACULUS_BOT_API_TOKEN'}")

HTML: Spring 2026 FutureEval Bot Tournament 03-21-2026.html
Output: Bot_Question_Tracking_2026-03-21_v01.csv
Auth: Token present


In [2]:
# === PARSE HTML SCORE TABLE ===
import re
import html as htmlmod

html_text = HTML_FILE.read_text(encoding="utf-8")
table_start = html_text.find('mb-3 w-full')
table_end = html_text.find('</table>', table_start) if html_text.find('</table>', table_start) > 0 else table_start + 400000
table_text = html_text[table_start:table_end]

# Auto-detect format: React JSON (escaped quotes) vs rendered HTML
if '\\"href\\"' in table_text or '\"href\"' in table_text:
    # React/RSC serialized JSON format (2026-03-21+)
    row_pattern = re.compile(
        r'\\"href\\":\\"\/questions\/(\d+)\\"'
        r',\\"children\\":\\"(.*?)\\"'
        r'.*?\\"children\\":\\"([\d.]+%?)\\"'
        r'.*?\\"children\\":\\"(-?[\d.]+|-?)\\"'
        r'.*?\\"children\\":\\"([\d.]+)\\"'
    )
    fmt = 'React JSON'
else:
    # Rendered HTML format (2026-03-07 and earlier)
    row_pattern = re.compile(
        r'href="https://www\.metaculus\.com/questions/(\d+)/">'
        r'(.*?)</a></td>'
        r'<th[^>]*>([^<]*)</th>'
        r'<td[^>]*>([^<]*)</td>'
        r'<th[^>]*>([^<]*)</th>'
    )
    fmt = 'Rendered HTML'

questions = []
for m in row_pattern.finditer(table_text):
    questions.append({
        'question_number': int(m.group(1)),
        'title': htmlmod.unescape(m.group(2).strip()),
        'coverage': m.group(3).strip(),
        'score': m.group(4).strip(),
        'question_weight': m.group(5).strip(),
    })

scored = [q for q in questions if q['score'] not in ('-', '')]
print(f"Format: {fmt}")
print(f"Parsed {len(questions)} questions ({len(scored)} scored, {len(questions) - len(scored)} unscored)")
if scored:
    scores = [float(q['score']) for q in scored]
    print(f"Score range: {min(scores):.3f} to {max(scores):.3f}, total: {sum(scores):.3f}")

Format: React JSON
Parsed 212 questions (47 scored, 165 unscored)
Score range: -14.535 to 79.287, total: 428.947


In [3]:
# === API FUNCTIONS ===
import requests
import json
import time

API_BASE = "https://www.metaculus.com/api2/questions"
MAX_RETRIES = 3
BACKOFF_BASE = 5


def fetch_question_data(question_id):
    """Fetch question data from Metaculus API with exponential backoff."""
    url = f"{API_BASE}/{question_id}/"
    headers = {'Authorization': f'Token {METACULUS_TOKEN}'} if METACULUS_TOKEN else {}
    for attempt in range(MAX_RETRIES):
        try:
            response = requests.get(url, headers=headers, timeout=30)
            if response.status_code == 200:
                return response.json()
            if response.status_code == 429:
                wait = BACKOFF_BASE * (2 ** attempt)
                print(f" [429, wait {wait}s]", end='')
                time.sleep(wait)
                continue
            response.raise_for_status()
        except requests.exceptions.RequestException as e:
            if attempt == MAX_RETRIES - 1:
                print(f" [ERROR: {e}]", end='')
                return None
            time.sleep(2)
    return None


def extract_fields(data):
    """Extract API-only fields: my_forecast, open_date, resolution_date, resolution_value, question_type."""
    question = data.get('question', {})
    q_type = question.get('type', '')

    # my_forecast
    my_forecast = 'Not Forecast'
    try:
        latest = question.get('my_forecasts', {}).get('latest', {})
        if latest:
            fv = latest.get('forecast_values', [])
            if q_type == 'binary' and len(fv) == 2:
                my_forecast = f"{fv[1]:.1%}"
            elif q_type == 'numeric':
                means = latest.get('means', [])
                my_forecast = f"{means[0]:.4f}" if means else (json.dumps(fv) if fv else 'Not Forecast')
            elif q_type == 'multiple_choice' and fv:
                options = question.get('options', [])
                if options and len(options) == len(fv):
                    my_forecast = '; '.join(f"{opt}: {v:.1%}" for opt, v in zip(options, fv))
                else:
                    my_forecast = json.dumps(fv)
    except Exception:
        pass

    # Dates
    open_time = data.get('open_time', '') or ''
    resolve_time = data.get('actual_resolve_time', '') or ''

    # Resolution
    resolution = question.get('resolution')
    if resolution is None:
        resolution_value = ''
    elif q_type == 'binary':
        resolution_value = 'Yes' if resolution == 1.0 else ('No' if resolution == 0.0 else str(resolution))
    else:
        resolution_value = str(resolution)

    return {
        'my_forecast': my_forecast,
        'open_date': open_time[:10],
        'resolution_date': resolve_time[:10],
        'resolution_value': resolution_value,
        'question_type': q_type,
    }


print("API functions defined")

API functions defined


In [4]:
# === BATCH FETCH ===
question_numbers = [q['question_number'] for q in questions]
api_results = {}
failed = []

print(f"Fetching {len(question_numbers)} questions (~{len(question_numbers) * RATE_LIMIT_DELAY / 60:.0f} min)...\n")

for i, qnum in enumerate(question_numbers, 1):
    if i % 10 == 1 or i == len(question_numbers):
        print(f"\n[{i}/{len(question_numbers)}] Q{qnum}", end='')
    else:
        print(".", end='')

    data = fetch_question_data(qnum)
    if data:
        api_results[qnum] = extract_fields(data)
    else:
        failed.append(qnum)

    if i < len(question_numbers):
        time.sleep(RATE_LIMIT_DELAY)

forecasted = sum(1 for r in api_results.values() if r['my_forecast'] != 'Not Forecast')
print(f"\n\nFetched: {len(api_results)} | Failed: {len(failed)} | With forecasts: {forecasted}")
if failed:
    print(f"Failed questions: {failed}")

Fetching 212 questions (~7 min)...


.........42235
.........Q41540
.........Q41848
.........Q42117
.........Q41665
.........Q41454
.........Q41750
.........Q41897
.........Q42036
.........Q42046
......... Q42107
......... Q42114
......... Q42233
......... Q42314
......... Q42358
......... Q42326
......... Q42504
......... Q42512
......... Q42562
......... Q42645
......... Q41901
[211/212] Q42248
[212/212] Q42321

Fetched: 212 | Failed: 0 | With forecasts: 186


In [5]:
# === MERGE + DISPLAY ===
import pandas as pd

df_html = pd.DataFrame(questions)
api_df = pd.DataFrame.from_dict(api_results, orient='index')
api_df.index.name = 'question_number'
api_df = api_df.reset_index()

df = df_html.merge(api_df, on='question_number', how='left')

# Fill NaN for failed API fetches
for col in ['my_forecast', 'open_date', 'resolution_date', 'resolution_value', 'question_type']:
    if col not in df.columns:
        df[col] = ''
    else:
        df[col] = df[col].fillna('')

# Column order
df = df[['question_number', 'title', 'coverage', 'score', 'question_weight',
         'my_forecast', 'open_date', 'resolution_date', 'resolution_value', 'question_type']]

# Sort: scored first (descending), then unscored
df['_sort'] = df['score'].apply(lambda s: (-float(s), 0) if s not in ('-', '') else (1e9, 0))
df = df.sort_values('_sort').drop(columns='_sort').reset_index(drop=True)

pd.set_option('display.max_rows', 220)
pd.set_option('display.max_colwidth', 60)
pd.set_option('display.width', 200)

print(f"Merged: {len(df)} questions | Scored: {(df['score'].apply(lambda s: s not in ('-', ''))).sum()} | "
      f"With forecasts: {(df['my_forecast'] != 'Not Forecast').sum()} | "
      f"Resolved: {(df['resolution_value'] != '').sum()}")
print(f"Type distribution: {df['question_type'].value_counts().to_dict()}")
print()
df

Merged: 212 questions | Scored: 47 | With forecasts: 186 | Resolved: 54
Type distribution: {'binary': 113, 'numeric': 61, 'multiple_choice': 25, 'discrete': 13}



,question_number,title,coverage,score,question_weight,my_forecast,open_date,resolution_date,resolution_value,question_type
0,42235,"Will the English Wikipedia have at least 7,145,000 artic...",100.0%,79.287,0.6,4.8%,2026-02-21,2026-03-09,no,binary
1,42095,Will any Individual Neutral Athlete (Russian and Belarus...,100.0%,59.838,1.0,61.5%,2026-02-14,2026-02-19,yes,binary
2,41835,Will the US government enter a shutdown before February ...,100.0%,40.187,1.0,32.5%,2026-01-23,2026-01-31,yes,binary
3,41845,What will be the total median number of global under-5 c...,100.0%,35.851,1.0,"[0.00594, 0.0060696456, 0.0061992912, 0.0063289367, 0.00...",2026-02-05,2026-03-18,4.858,numeric
4,42316,Which film will win Best Picture at the 98th Academy Awa...,100.0%,25.532,1.0,Bugonia: 2.5%; Frankenstein: 1.8%; Sinners: 15.8%; Other...,2026-03-02,2026-03-20,Other,multiple_choice
5,41894,How many Oscars will Sinners win at the 2026 Academy Awa...,100.0%,23.992,1.0,Not Forecast,2026-02-02,2026-03-16,4.0,discrete
6,41846,Will there be a successful coup in Africa or Latin Ameri...,100.0%,23.159,1.0,13.5%,2026-02-06,2026-03-01,no,binary
7,42116,Will CDU win the most seats in the Baden-Württemberg Lan...,100.0%,22.489,1.0,72.5%,2026-02-19,2026-03-10,no,binary
8,41871,What will be NVIDIA's forward guidance in their Q4 FY202...,100.0%,21.953,0.7,"[0.01088, 0.0718799877, 0.0740801913, 0.0762803948, 0.07...",2026-01-26,2026-02-25,7500000000.0,numeric
9,42507,Will the Global Polio Eradication Initiative (GPEI) repo...,100.0%,20.103,1.0,77.5%,2026-03-09,2026-03-18,yes,binary


In [6]:
# === WRITE CSV ===
df.to_csv(OUTPUT_CSV, index=False)
size_kb = OUTPUT_CSV.stat().st_size / 1024
print(f"Wrote {len(df)} rows to {OUTPUT_CSV.name} ({size_kb:.1f} KB)")

Wrote 212 rows to Bot_Question_Tracking_2026-03-21_v01.csv (187.7 KB)
